# Time-relative features

This experiment tests whether within-time-step percentile ranks improve robustness to temporal changes in anonymised transaction features. The comparison uses the same fixed HistGradientBoosting configuration for raw features and raw features augmented with percentile ranks, isolating the effect of the representation.

Ranks are calculated without labels, using all transactions in each time step. This is a **batch-transductive** transformation: it assumes the full contemporaneous batch is available and would not be directly applicable to individual transactions arriving in real time. Four chronological validation windows are evaluated; no submission is generated.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'raw'
train = pd.read_csv(DATA / 'train.csv')
test = pd.read_csv(DATA / 'test.csv')
features = [c for c in train.columns if c.startswith('feat_')]
labelled = train.loc[train['label'].notna()].copy()
assert set(labelled['label'].unique()) <= {0, 1}
assert 'time_step' in train and 'time_step' in test
assert len(features) > 0
print('Rows:', len(train), 'labelled:', len(labelled), 'test:', len(test), 'features:', len(features))

Rows: 141772 labelled: 31235 test: 15329 features: 165


## Within-time-step percentile features

For each numerical feature, compute its percentile rank among transactions in the same time step, including unlabelled rows. Concatenating train and test for this label-free transformation ensures consistent row alignment. Transaction IDs and labels are excluded.


In [2]:
# Use position-based alignment so repeated txIds/index values cannot silently corrupt features.
all_rows = pd.concat([train[['time_step'] + features], test[['time_step'] + features]],
                     ignore_index=True)
rank_features = [f'{c}__step_pct' for c in features]
step_ranks = all_rows.groupby('time_step', sort=False)[features].rank(pct=True, method='average')
step_ranks.columns = rank_features
train_aug = train.reset_index(drop=True).copy()
test_aug = test.reset_index(drop=True).copy()
for c in rank_features:
    train_aug[c] = step_ranks.iloc[:len(train)][c].to_numpy(dtype=np.float32)
    test_aug[c] = step_ranks.iloc[len(train):][c].to_numpy(dtype=np.float32)
labelled_aug = train_aug.loc[train_aug['label'].notna()].copy()
assert len(labelled_aug) == len(labelled)
assert np.array_equal(labelled_aug['label'].to_numpy(), labelled['label'].to_numpy())
assert all(train_aug[c].dropna().between(0,1).all() for c in rank_features)
print('Original features:', len(features), 'Augmented:', len(features + rank_features))

/var/folders/cn/63qn1bxd4ld51574vmz6qx7m0000gn/T/ipykernel_69514/3885106433.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_aug[c] = step_ranks.iloc[:len(train)][c].to_numpy(dtype=np.float32)
/var/folders/cn/63qn1bxd4ld51574vmz6qx7m0000gn/T/ipykernel_69514/3885106433.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_aug[c] = step_ranks.iloc[len(train):][c].to_numpy(dtype=np.float32)


Original features: 165 Augmented: 330


## Chronological validation

Both models use identical hyperparameters and train only on labelled observations preceding each validation window. Two longer windows simulate a larger gap between training history and evaluation, while two recent windows test performance nearer the end of the labelled period.


In [ ]:
HGB_PARAMS = dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
                  l2_regularization=1.0, random_state=42)
WINDOWS = [
    ('long_15_28', 15, 28),
    ('long_22_35', 22, 35),
    ('recent_25_29', 25, 29),
    ('recent_30_35', 30, 35),
]

def run_window(name, lo, hi):
    tr = labelled_aug[labelled_aug.time_step < lo]
    va = labelled_aug[labelled_aug.time_step.between(lo, hi)]
    if tr.empty or va.empty: raise ValueError(f'Empty split {name}')
    ytr = tr.label.astype(int).to_numpy()
    yva = va.label.astype(int).to_numpy()
    if len(np.unique(ytr)) != 2 or len(np.unique(yva)) != 2:
        raise ValueError(f'Only one class in {name}')
    out = dict(window=name, train_rows=len(tr), valid_rows=len(va),
               fraud_rate=yva.mean())
    for tag, cols in [('raw', features), ('raw_plus_percentile', features + rank_features)]:
        model = HistGradientBoostingClassifier(**HGB_PARAMS)
        model.fit(tr[cols], ytr)
        p = model.predict_proba(va[cols])[:, 1]
        out[f'{tag}_pr_auc'] = average_precision_score(yva, p)
        out[f'{tag}_roc_auc'] = roc_auc_score(yva, p)
    out['delta_pr_auc'] = out['raw_plus_percentile_pr_auc'] - out['raw_pr_auc']
    out['delta_roc_auc'] = out['raw_plus_percentile_roc_auc'] - out['raw_roc_auc']
    print(f"{name}: ROC-AUC raw={out['raw_roc_auc']:.6f} augmented={out['raw_plus_percentile_roc_auc']:.6f} delta={out['delta_roc_auc']:+.6f}; PR-AUC delta={out['delta_pr_auc']:+.6f}", flush=True)
    return out

## Validation results

Report both ROC-AUC, the Kaggle competition metric, and average precision (PR-AUC), which was used in the original feature comparison. The two metrics may favour different representations; this notebook does not change the original training procedure.


In [ ]:
results = pd.DataFrame([run_window(*window) for window in WINDOWS])
display(results.round(6))
print('ROC-AUC improved windows:', int((results.delta_roc_auc > 0).sum()), '/', len(results))
print('PR-AUC improved windows:', int((results.delta_pr_auc > 0).sum()), '/', len(results))
print('Long-horizon deltas:', results.loc[results.window.str.startswith('long'), ['window','delta_roc_auc','delta_pr_auc']].to_string(index=False))
print('Recent-window deltas:', results.loc[results.window.str.startswith('recent'), ['window','delta_roc_auc','delta_pr_auc']].to_string(index=False))

## Findings

The original comparison found no consistent improvement from adding within-step percentiles: average precision declined on three of the four validation windows. ROC-AUC is now reported alongside it so that the feature change can also be assessed against the competition metric after rerunning the evaluation.

The experiment is a controlled feature ablation using HistGradientBoosting, not a direct comparison against the final chronological stack. Even if percentile features improve an individual validation window, that would not establish better performance on the unseen test period. Their dependence on full time-step batches also limits applicability to real-time deployment.
